<a href="https://colab.research.google.com/github/nicosesma/otc_strategies/blob/main/BTC_STmom_TSL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from binance.client import Client
client = Client()

In [1]:
def getdata(symbol, start):
    frame = pd.DataFrame(client.get_historical_klines(symbol,
                                                     '1h',
                                                     start))
    frame = frame.iloc[:,:6]
    frame.columns = ['Time','Open','High','Low','Close','Volume']
    frame.set_index('Time', inplace=True)
    frame.index = pd.to_datetime(frame.index,unit='ms')
    frame = frame.astype(float)
    return frame

In [ ]:
df = getdata('ETHUSDT','2023-01-01')

In [ ]:
df['price'] = df.Open.shift(-1)

In [ ]:
df['ret'] = df.Close.pct_change()

In [ ]:
df = df.dropna()

In [ ]:
def trail(df,entry, dist):
    profits = []
    in_position = False

    for index,row in df.iterrows():
        if not in_position and row.ret > entry:
            buyprice = row.price
            in_position = True
            trailing_stop = buyprice * dist
        if in_position:
            if row.Close * dist >= trailing_stop:
                trailing_stop = row.Close * dist
            if row.Close <= trailing_stop:
                sellprice = row.price
                profit = (sellprice-buyprice)/buyprice - 0.0015
                profits.append(profit)
                in_position = False
    return (pd.Series(profits) + 1).cumprod()

In [ ]:
trail(df,0.005,0.94)

0     1.286272
1     1.279494
2     1.277389
3     1.341498
4     1.272401
5     1.148793
6     1.325860
7     1.394919
8     1.380283
9     1.583146
10    1.459898
11    1.429063
12    1.382261
13    1.419192
14    1.347756
15    1.331189
16    1.237136
17    1.378575
18    1.393294
19    1.304050
20    1.252303
21    1.188985
22    1.212334
23    1.144386
24    1.450356
25    1.414380
26    1.404769
27    1.599718
28    1.539659
29    1.550753
30    1.574187
31    1.537147
32    1.730842
33    1.671261
34    1.568195
35    1.471905
dtype: float64

In [ ]:
(pd.Series(profits) + 1).prod()

1.8470690288140352

In [ ]:
43070.25/16551.47

2.6022008921261977

In [ ]:
len(profits)

82

In [ ]:
2307.36/1196.02

1.9291985083861476

In [ ]:
df

,Open,High,Low,Close,Volume,price,ret
Time,,,,,,,
2023-01-01 01:00:00,1194.09,1196.37,1193.84,1196.02,3157.2079,1196.01,0.001616
2023-01-01 02:00:00,1196.01,1196.74,1194.11,1195.40,3752.0476,1195.41,-0.000518
2023-01-01 03:00:00,1195.41,1195.41,1191.71,1194.04,7493.4207,1194.05,-0.001138
2023-01-01 04:00:00,1194.05,1194.05,1190.57,1192.92,6409.2491,1192.92,-0.000938
2023-01-01 05:00:00,1192.92,1194.67,1192.71,1194.54,2316.3448,1194.55,0.001358
...,...,...,...,...,...,...,...
2024-02-03 09:00:00,2308.30,2316.39,2308.30,2315.12,3393.9487,2315.11,0.002950
2024-02-03 10:00:00,2315.11,2315.12,2310.80,2314.12,4332.8931,2314.12,-0.000432
2024-02-03 11:00:00,2314.12,2314.13,2300.24,2303.89,7495.5710,2303.88,-0.004421
